# IMDC 2026 — Data Audit

Phase 1 checks before any modeling: fold boundaries, leakage guards, structural sanity, completeness, and missingness across all raw tables. These checks back the assertions in `src/imdc/data/validate.py` and the pytest suite in `tests/test_folds_and_leakage.py`.

In [1]:
import pandas as pd
import numpy as np

from imdc.config import ALL_UFS, MANDATORY_UFS, EXCLUDED_UF, DENGUE_TARGET_CITIES, CHIKUNGUNYA_TARGET_CITIES
from imdc.data.loaders import (
    load_cases, load_climate, load_forecasting_climate, load_ocean_indices,
    load_environ_vars, load_population, load_access_afya, load_geo_map,
)
from imdc.data.folds import get_folds, get_forecast_origin, cutoff_filter, cutoff_filter_forecasting_climate
from imdc.data.validate import (
    assert_no_leakage, assert_gap_weeks_absent, assert_all_sundays,
    assert_no_duplicate_series, assert_non_negative,
)

pd.set_option("display.max_columns", 50)

## 1. Fold boundaries (derived from data, not hardcoded)

In [2]:
dengue_folds = get_folds("dengue")
chik_folds = get_folds("chikungunya")

fold_summary = pd.DataFrame([
    {"disease": "dengue", "fold": f.id, "train_cutoff": f.train_cutoff.date(),
     "target_start": f.target_start.date(), "target_end": f.target_end.date(),
     "gap_weeks": (f.target_start - f.train_cutoff).days // 7 - 1}
    for f in dengue_folds
] + [
    {"disease": "chikungunya", "fold": f.id, "train_cutoff": f.train_cutoff.date(),
     "target_start": f.target_start.date(), "target_end": f.target_end.date(),
     "gap_weeks": (f.target_start - f.train_cutoff).days // 7 - 1}
    for f in chik_folds
])
print("Dengue forecast origin (latest observed date):", get_forecast_origin("dengue").date())
print("Chikungunya forecast origin:", get_forecast_origin("chikungunya").date())
fold_summary

Dengue forecast origin (latest observed date): 2026-03-08


Chikungunya forecast origin: 2026-03-08


,disease,fold,train_cutoff,target_start,target_end,gap_weeks
0,dengue,1,2022-06-19,2022-10-09,2023-10-01,15
1,dengue,2,2023-06-18,2023-10-08,2024-09-29,15
2,dengue,3,2024-06-16,2024-10-06,2025-09-28,15
3,dengue,4,2025-06-15,2025-10-05,2026-03-08,15
4,chikungunya,1,2022-06-19,2022-10-09,2023-10-01,15
5,chikungunya,2,2023-06-18,2023-10-08,2024-09-29,15
6,chikungunya,3,2024-06-16,2024-10-06,2025-09-28,15
7,chikungunya,4,2025-06-15,2025-10-05,2026-03-08,15


## 2. Leakage checks

Confirms: (a) the naive `~target_N` filter leaks the unflagged gap weeks, (b) an explicit `date <= train_cutoff` filter does not, and (c) auxiliary tables with no fold flags at all are safe once the same explicit cutoff is applied.

In [3]:
dengue = load_cases("dengue")
climate = load_climate()
ocean = load_ocean_indices()
fc = load_forecasting_climate()
access = load_access_afya()

for fold in dengue_folds:
    # naive filter -> should leak the gap weeks
    naive = dengue[~dengue[f"target_{fold.id}"]]
    try:
        assert_gap_weeks_absent(naive, fold)
        leaked = False
    except AssertionError:
        leaked = True

    # explicit cutoff filter -> should NOT leak, for every table
    d_ok = cutoff_filter(dengue, fold.train_cutoff)
    c_ok = cutoff_filter(climate, fold.train_cutoff)
    o_ok = cutoff_filter(ocean, fold.train_cutoff)
    a_ok = cutoff_filter(access, fold.train_cutoff, date_col="access_date")
    fc_ok = cutoff_filter_forecasting_climate(fc, fold.train_cutoff)

    assert_gap_weeks_absent(d_ok, fold)
    assert_no_leakage(d_ok, fold.train_cutoff, name=f"fold{fold.id} dengue")
    assert_no_leakage(c_ok, fold.train_cutoff, name=f"fold{fold.id} climate")
    assert_no_leakage(o_ok, fold.train_cutoff, name=f"fold{fold.id} ocean")
    assert_no_leakage(a_ok, fold.train_cutoff, date_col="access_date", name=f"fold{fold.id} access_afya")

    print(f"fold {fold.id}: naive ~target_N filter leaks gap weeks = {leaked} | "
          f"explicit cutoff filter leaks = False (all tables clean)")

fold 1: naive ~target_N filter leaks gap weeks = True | explicit cutoff filter leaks = False (all tables clean)


fold 2: naive ~target_N filter leaks gap weeks = True | explicit cutoff filter leaks = False (all tables clean)


fold 3: naive ~target_N filter leaks gap weeks = True | explicit cutoff filter leaks = False (all tables clean)


fold 4: naive ~target_N filter leaks gap weeks = True | explicit cutoff filter leaks = False (all tables clean)


## 3. Structural sanity

Dates all Sundays, no duplicate (geocode, date) rows, no negative case counts, for both diseases.

In [4]:
chik = load_cases("chikungunya")

for name, df in [("dengue", dengue), ("chikungunya", chik)]:
    assert_all_sundays(df, name=name)
    assert_no_duplicate_series(df, ["geocode", "date"], name=name)
    assert_non_negative(df, "casos", name=name)
    print(f"{name}: OK - all Sundays, no dupes, no negatives. "
          f"{df['geocode'].nunique()} municipalities, {df['uf'].nunique()} UFs, "
          f"{df['date'].min().date()} to {df['date'].max().date()}")

missing_ufs = set(ALL_UFS) - set(dengue["uf"].unique())
print("\nUFs missing from dengue data (should be empty):", missing_ufs)
print("ES present in raw data (must be excluded downstream for mandatory target):", "ES" in dengue["uf"].unique())

dengue: OK - all Sundays, no dupes, no negatives. 5570 municipalities, 27 UFs, 2010-01-03 to 2026-03-08


chikungunya: OK - all Sundays, no dupes, no negatives. 5570 municipalities, 27 UFs, 2013-12-29 to 2026-03-08

UFs missing from dengue data (should be empty): set()
ES present in raw data (must be excluded downstream for mandatory target): True


## 4. Completeness — municipality × week grid coverage

For each table, check whether every municipality has a row for every expected epidemiological week in its date range (gaps here would need explicit handling before feature engineering, e.g. reindex + fill).

In [5]:
def grid_completeness(df, date_col, geo_col="geocode", name="table"):
    n_weeks_expected = df[date_col].nunique()
    n_geo = df[geo_col].nunique()
    expected = n_weeks_expected * n_geo
    actual = len(df)
    print(f"{name}: {actual:,} rows vs {expected:,} expected (n_weeks x n_geo) "
          f"-> {actual / expected:.1%} complete, {n_geo} unique geocodes, {n_weeks_expected} unique weeks")
    # per-municipality week counts, flag outliers
    counts = df.groupby(geo_col)[date_col].nunique()
    print(f"  weeks-per-municipality: min={counts.min()}, median={counts.median():.0f}, max={counts.max()}")
    return counts

_ = grid_completeness(dengue, "date", name="dengue")
_ = grid_completeness(chik, "date", name="chikungunya")
_ = grid_completeness(climate, "date", name="climate")
_ = grid_completeness(access, "access_date", name="access_afya")

dengue: 4,706,650 rows vs 4,706,650 expected (n_weeks x n_geo) -> 100.0% complete, 5570 unique geocodes, 845 unique weeks


  weeks-per-municipality: min=845, median=845, max=845
chikungunya: 3,548,090 rows vs 3,548,090 expected (n_weeks x n_geo) -> 100.0% complete, 5570 unique geocodes, 637 unique weeks
  weeks-per-municipality: min=637, median=637, max=637


climate: 7,621,223 rows vs 7,621,223 expected (n_weeks x n_geo) -> 100.0% complete, 5567 unique geocodes, 1369 unique weeks


  weeks-per-municipality: min=1369, median=1369, max=1369
access_afya: 8,963,730 rows vs 8,963,718 expected (n_weeks x n_geo) -> 100.0% complete, 4534 unique geocodes, 1977 unique weeks


  weeks-per-municipality: min=1977, median=1977, max=1977


## 5. Missingness patterns

NaN counts per column across the auxiliary tables, and cross-table geocode-set consistency (do dengue, chikungunya, climate, environ_vars, population all reference the same 5,570 municipalities?).

In [6]:
environ = load_environ_vars()
population = load_population()
geo_map = load_geo_map()

for name, df in [("dengue", dengue), ("chikungunya", chik), ("climate", climate),
                  ("forecasting_climate", fc), ("access_afya", access),
                  ("environ_vars", environ), ("population", population)]:
    na_counts = df.isna().sum()
    na_counts = na_counts[na_counts > 0]
    print(f"{name}: {'no NaNs' if na_counts.empty else dict(na_counts)}")

geocode_sets = {
    "dengue": set(dengue["geocode"].unique()),
    "chikungunya": set(chik["geocode"].unique()),
    "climate": set(climate["geocode"].unique()),
    "environ_vars": set(environ["geocode"].unique()),
    "population": set(population["geocode"].unique()),
    "geo_map": set(geo_map["geocode"].unique()),
}
base = geocode_sets["dengue"]
print("\nGeocode set consistency (vs dengue's set):")
for name, s in geocode_sets.items():
    print(f"  {name}: {len(s)} geocodes, {len(base - s)} missing from it, {len(s - base)} extra")

dengue: no NaNs
chikungunya: no NaNs


climate: no NaNs
forecasting_climate: {'temp_med': np.int64(1080), 'umid_med': np.int64(1080), 'precip_tot': np.int64(1080)}


access_afya: no NaNs
environ_vars: no NaNs
population: no NaNs

Geocode set consistency (vs dengue's set):
  dengue: 5570 geocodes, 0 missing from it, 0 extra
  chikungunya: 5570 geocodes, 0 missing from it, 0 extra
  climate: 5567 geocodes, 3 missing from it, 0 extra
  environ_vars: 5570 geocodes, 0 missing from it, 0 extra
  population: 5571 geocodes, 0 missing from it, 1 extra
  geo_map: 5570 geocodes, 0 missing from it, 0 extra


## 6. Target-city and ES exclusion sanity

In [7]:
dengue_flagged_cities = set(dengue.loc[dengue["target_city"], "geocode"].unique())
chik_flagged_cities = set(chik.loc[chik["target_city"], "geocode"].unique())

print("Dengue target cities match config:", dengue_flagged_cities == set(DENGUE_TARGET_CITIES),
      f"({len(dengue_flagged_cities)} cities)")
print("Chikungunya target cities match config:", chik_flagged_cities == set(CHIKUNGUNYA_TARGET_CITIES),
      f"({len(chik_flagged_cities)} cities)")

es_geocodes = dengue.loc[dengue["uf"] == "ES", "geocode"].nunique()
print(f"\nES municipalities present in raw data: {es_geocodes} (must be dropped for mandatory state target)")
print("MANDATORY_UFS count (should be 26):", len(MANDATORY_UFS))

Dengue target cities match config: True (15 cities)
Chikungunya target cities match config: True (10 cities)



ES municipalities present in raw data: 78 (must be dropped for mandatory state target)
MANDATORY_UFS count (should be 26): 26


## 7. Follow-up on minor anomalies found above

- **3 municipalities missing entirely from `climate.csv.gz`**: geocodes 2916104, 2919926 (BA), 2605459 (PE). Same geocode 2605459 is also the only one with NaNs in `forecasting_climate` (1080 rows). Negligible for state-level aggregation (3 of 5570 municipalities) but must be handled explicitly (e.g. drop from population-weighted state climate average) rather than silently propagating NaN.
- **1 extra geocode in `population.csv.gz`** (5101837) not present in dengue/chikungunya/geo_map/environ_vars, only for year 2025 with population 5,877 — looks like a newly created municipality (Brazil occasionally splits municipalities). No case data exists for it, so it's irrelevant to modeling; safe to ignore.
- **12 duplicate `(geocode, access_date)` rows in `access_afya`**, with differing `access_count` values (not exact dupes) — must be deduplicated (sum access_count) before use as a feature; the raw table cannot be joined 1:1 on `(geocode, date)` as-is.

## Summary

Findings from this audit should be logged here after running, and referenced by the feature-engineering and paper data-description sections. Key structural facts confirmed above: fold boundaries derived correctly, gap-week leakage trap reproduced and guarded against, no structural anomalies (dates/dupes/negatives) in case data, target-city and ES handling correct.